# Multi-speaker dialogue generation with FireRedTTS‑2 and OpenVINO

FireRedTTS‑2 is a long-form streaming TTS system for multi-speaker dialogue generation, delivering stable, natural speech with reliable speaker switching and context-aware prosody. It is highlighted by following features:
- **Long Conversational Speech Generation**: It currently supports 3 minutes dialogues with 4 speakers and can be easily scaled to longer conversations
with more speakers by extending training corpus.
- **Multilingual Support**: It supports multiple languages including English, Chinese, Japanese, Korean, French, German, and Russian. Support zero-shot voice cloning for cross-lingual and code-switching scenarios.
- **Ultra-Low Latency**: Building on the new **12.5Hz streaming** speech tokenizer, we employ a dual-transformer architecture that operates on a text–speech interleaved sequence, enabling flexible sentence-bysentence generation and reducing first-packet latency，Specifically, on an L20 GPU, our first-packet latency as low as 140ms while maintaining high-quality audio output.
- **Strong Stability**：Our model achieves high similarity and low WER/CER in both monologue and dialogue tests.
- **Random Timbre Generation**:Useful for creating ASR/speech interaction data.

More details can be found in the [paper](https://arxiv.org/abs/2509.02020), original [repository](https://github.com/FireRedTeam/FireRedTTS2) and [model card](https://huggingface.co/FireRedTeam/FireRedTTS2)

In this tutorial we consider how to run and optimize FireRedTTS‑2 using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
- [Create Inference Pipeline](#Create-Inference-Pipeline)
    - [Select Inference Device](#Select-Inference-Device)
    - [Run Dialogue Generation](#Run-Dialogue-Generation)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/janus-multimodal-generation/janus-multimodal-generation.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
# Fetch `notebook_utils` module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("firetts2.ipynb")

In [2]:
from cmd_helper import clone_repo
from pip_helper import pip_install
import platform

!pip uninstall -y FireRedTTS2

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "torchaudio==2.7.1",
    "nncf",
    "openvino>=2025.3.0",
    "gradio",
)

repo_dir = Path("FireRedTTS2")
revision = "bfacbfb7bb88cade9c0b9ab2644ebd7f75c6989c"
clone_repo("https://github.com/openvino-dev-samples/FireRedTTS2.git", revision)

pip_install(
    "-q -e",
    str(repo_dir),
)

pip_install(
    "-q -r",
    str(repo_dir / "requirements.txt"),
)
if platform.system() == "Darwin":
    pip_install("numpy<2.0")

Found existing installation: fireredtts2 0.1
Uninstalling fireredtts2-0.1:
  Successfully uninstalled fireredtts2-0.1



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: switching to 'bfacbfb7bb88cade9c0b9ab2644ebd7f75c6989c'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at bfacbfb Update llm.py

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

 Janus is PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). [OpenVINO model conversion API](https://docs.openvino.ai/2024/openvino-workflow/model-preparation.html#convert-a-model-with-python-convert-model) should be used for these purposes. `ov.convert_model` function accepts original PyTorch model instance and example input for tracing and returns `ov.Model` representing this model in OpenVINO framework. Converted model can be used for saving on disk using `ov.save_model` function or directly loading on device using `core.complie_model`. 

The script `ov_firetts_helper.py` contains helper function for model conversion.

In [3]:
from ov_fireredtts_helper import convert_fireredtts2

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("fireredtts2.ipynb")

pt_model_path = Path("pretrained_models")
if not pt_model_path.exists():
    !git clone https://huggingface.co/FireRedTeam/FireRedTTS2 pretrained_models

model_path = "FireRedTTS2-ov"
convert_fireredtts2(pt_model_path, model_path)

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cpu for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


⌛ pretrained_models conversion started. Be patient, it may takes some time.
⌛ Load Original model
🔍 Detected Configuration:
  num_heads: 12
  num_kv_heads: 2
  dim: 1536
  head_dim: 128
  intermediate_size: 8960
  num_layers: 28
  max_seq_len: 4096
  tie_word_embeddings: True

🔧 Removing 'model.' prefix...

🔑 Cleaned key examples:
  layers.0.self_attn.q_proj.weight
  layers.0.self_attn.q_proj.bias
  layers.0.self_attn.k_proj.weight
  layers.0.self_attn.k_proj.bias
  layers.0.self_attn.v_proj.weight

⚠️  Missing keys: ['embed_tokens.weight']

✅ Conversion completed!
🔍 Detected Configuration:
  num_heads: 12
  num_kv_heads: 2
  dim: 1536
  head_dim: 128
  intermediate_size: 8960
  num_layers: 4
  max_seq_len: 4096
  tie_word_embeddings: True

🔧 Removing 'model.' prefix...

🔑 Cleaned key examples:
  layers.0.self_attn.q_proj.weight
  layers.0.self_attn.q_proj.bias
  layers.0.self_attn.k_proj.weight
  layers.0.self_attn.k_proj.bias
  layers.0.self_attn.v_proj.weight

⚠️  Missing keys: ['em

/home2/ethan/intel/openvino_notebooks/notebooks/fireredtts2/ov_fireredtts_helper.py:760: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  vq_out_length = torch.tensor(


vq_out_feats shape: 1
vq_out_feats shape: 1
✅ AUDIO_UPSAMPLER model successfully converted
⌛ Convert AUDIO_DECODER model


/home2/ethan/intel/openvino_notebooks/notebooks/fireredtts2/FireRedTTS2/fireredtts2/codec/utils.py:7: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  max_len = max_len if max_len > 0 else lengths.max().item()
/home2/ethan/intel/openvino_notebooks/notebooks/fireredtts2/FireRedTTS2/fireredtts2/codec/utils.py:26: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  num_blocks = torch.ceil(torch.tensor(attn_mask.shape[1] / chunk_size)).to(torch.int64)
/home2/ethan/intel/openvino_notebooks/notebooks/fireredtts2/FireRedTTS2/firer

...

/home2/ethan/intel/openvino_notebooks/notebooks/fireredtts2/FireRedTTS2/fireredtts2/codec/model.py:221: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  audio16k_length = torch.tensor(
/home2/ethan/intel/openvino_notebooks/notebooks/fireredtts2/FireRedTTS2/fireredtts2/codec/whisper.py:330: TracerWarning: torch.from_numpy results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  mel_filters = torch.from_numpy(self.mel_filters).type(torch.float32).to(device)
/home2/ethan/intel/openvino_notebooks/notebooks/fireredtts2/Fire

✅ AUDIO_ENCODER model successfully converted
⌛ Convert DECODER_MODEL model


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
/home2/ethan/intel/openvino_notebooks/openvino_venv/lib/python3.10/site-packages/transformers/cache_utils.py:568: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  or not self.key_cache[layer_idx].numel()  # the layer has no cache
/home2/ethan/intel/openvino_notebooks/notebooks/fireredtts2/ov_fireredtts_helper.py:371: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/home2/ethan/i

✅ Decoder model successfully converted
⌛ Convert BACKBONE_MODEL model
✅ Backbone model successfully converted
✅ pretrained_models model conversion finished. You can find results in FireRedTTS2-ov


PosixPath('FireRedTTS2-ov')

## Create Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

`OVFireRedTTS2` defined in `ov_fireredtts_helper.py` provides unified interface for running model inference. It accepts model directory and target device for inference.

### Select Inference Device
[back to top ⬆️](#Table-of-contents:)

In [4]:
from notebook_utils import device_widget

device = device_widget("CPU", ["NPU"])

device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

`OVFireRedTTS2` class used for pre- and postprocessing steps in original FireRedTTS-2 model. Our model is also compatible with the same processor code and we can reuse it. 

ℹ️ **Limitation**
- Currently it can support `dialogue mode` only. 
- Codec model can be deployed to `CPU` only.

In [6]:
from ov_fireredtts_helper import OVFireRedTTS2

ov_model = OVFireRedTTS2(model_path, gen_type="dialogue", device=device.value, codec_device="CPU")

RuntimeError: Exception from src/inference/src/cpp/core.cpp:134:
Exception from src/inference/src/dev/plugin.cpp:58:
Exception from src/core/src/pass/graph_rewrite.cpp:298:
[FuseBinaryEltwise] END: node: opset1::Add Add_494266 (SnippetsOpset::BrgemmCPU MatMul_494263[0]:f32[?,20,?,?], opset1::Parameter Add_494266[0]:f32[1,1,1,300]) -> (f32[?,20,?,300]) CALLBACK HAS THROWN: Exception from src/core/src/dimension.cpp:227:
Cannot get length of dynamic dimension






### Run visual language chat
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import torchaudio


text_list = [
    "[S1]It's alright, we'll take a breath and plan the next pass together.",
    "[S2]Yeah, thanks. We'll get it right this time.",
    "[S1]Let's review our signals tonight so we're in sync on the field tomorrow.",
]
prompt_wav_list = [
    "FireRedTTS2/examples/chat_prompt/en/S1.flac",
    "FireRedTTS2/examples/chat_prompt/en/S2.flac",
]

prompt_text_list = [
    "[S1]I think we should just talk about what happened and move on because there's going to be other jousts and Sir Saif isn't done yet. It's not, he's not, it's not done yet.",
    "[S2]You know, maybe sorry, maybe maybe I pushed, maybe I pushed too hard. I was really excited. I didn't mean to make you snap.",
]

all_audio = ov_model.generate_dialogue(
    text_list=text_list,
    prompt_wav_list=prompt_wav_list,
    prompt_text_list=prompt_text_list,
    temperature=0.9,
    topk=30,
)
torchaudio.save("chat_clone_ov.wav", all_audio, 24000)

In [ ]:
import IPython

display(IPython.display.Audio("chat_clone_ov.wav"))

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/